# Experiment 13: Model-Wide Adaptive Bespoke Tucker Compression

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Innovations:
1. **Automated Bespoke Rank Selection via HOSVD Energy Profiling**:
   Eliminates rigid uniform rank assignment. For each individual tensor $\mathcal{T}_l^{(s)} \in \mathbb{R}^{6 \times 400 \times 1152}$:
   - Computes singular values of mode unfoldings: $T_{(1)}, T_{(2)}, T_{(3)}$.
   - Evaluates cumulative energy curves $E_n(r) = \frac{\sum_{i=1}^r (s_i^{(n)})^2}{\sum_{i=1}^{I_n} (s_i^{(n)})^2}$.
   - Selects bespoke optimal ranks along each mode adaptively via target energy cutoff $\tau$:
     $$R_n = \min \{ r : E_n(r) \ge \tau \}$$
   - Highly redundant middle layers automatically drop to lower ranks ($R_2 \approx 140, R_3 \approx 300$), while boundary representation layers retain higher ranks ($R_2 \approx 220, R_3 \approx 450$).
2. **Dedicated Independent Sublayer DBSCAN Clustering**:
   All 78 projection matrices are clustered independently using their own empirical activation profiles, with superweights isolated in FP32.
3. **Multi-Tier Adaptive Sweeps**:
   - **Adaptive Sweet Spot ($\tau = 0.65$)**: Targets ~76–78% reconstruction error across every individual tensor.
   - **Adaptive High-Fidelity ($\tau = 0.70$)**: Targets ~71–73% reconstruction error.
   - **Adaptive Aggressive ($\tau = 0.60$)**: Targets ~80–82% reconstruction error.
4. **Memory Safety & Zero-VRAM Optimization**:
   All decomposition and Adam GD steps execute on CPU (`device="cpu"`). Inference uses `logits_to_keep=1`.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded neural_decomp library.
Device: NVIDIA GeForce RTX 3070 Ti | CUDA Available: True


In [2]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer (Unprocessed Clean Baseline)
# =====================================================================
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
D_IN = model.model.layers[0].mlp.gate_proj.weight.shape[1]
D_OUT = model.model.layers[0].mlp.gate_proj.weight.shape[0]

print(f"Loaded {model_id}: {NUM_LAYERS} Transformer Decoder Layers")
print(f"MLP Dimensions: in_features={D_IN}, intermediate_features={D_OUT}")

# Cache pristine weights on CPU across all 26 layers for calibration & rollback
W_orig_all = {
    l: {
        "gate_proj": model.model.layers[l].mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   model.model.layers[l].mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": model.model.layers[l].mlp.down_proj.weight.data.clone().cpu(),
    }
    for l in range(NUM_LAYERS)
}
print(f"Cached pristine weights on CPU for all {NUM_LAYERS} layers (78 projection matrices).")

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 372.57it/s]


Loaded google/gemma-3-1b-it: 26 Transformer Decoder Layers
MLP Dimensions: in_features=1152, intermediate_features=6912
Cached pristine weights on CPU for all 26 layers (78 projection matrices).
time: 5.53s
cummulative_time: 7.36s


In [ ]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

Loaded GLUE MNLI: 9,815 total samples | Active Evaluation Subset: 150 samples
time: 3.61s
cummulative_time: 10.98s


## Step 1: Simultaneous Tri-Hook Profiling Across All 26 Layers

We register forward hooks on `gate_proj` (`act_fn`), `up_proj`, and `down_proj` input across all 26 layers during baseline inference to record distinct activation trajectories.

In [4]:
# =====================================================================
# STEP 4: Simultaneous Tri-Hook Profiling & Baseline MNLI Inference
# =====================================================================
layer_trajectories = {
    l: {"gate": [], "up": [], "down": []} for l in range(NUM_LAYERS)
}
current_acts = {
    l: {"gate": None, "up": None, "down": None} for l in range(NUM_LAYERS)
}

def make_hook(layer_idx, sub_key, is_input=False):
    def hook_fn(module, input_tensor, output_tensor):
        t = input_tensor[0] if is_input else (output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor)
        current_acts[layer_idx][sub_key] = t.detach().cpu()
    return hook_fn

hooks = []
for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp
    hooks.append(lmod.act_fn.register_forward_hook(make_hook(l, "gate", is_input=False)))
    hooks.append(lmod.up_proj.register_forward_hook(make_hook(l, "up", is_input=False)))
    hooks.append(lmod.down_proj.register_forward_hook(make_hook(l, "down", is_input=True)))

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Simultaneous Tri-Hook Profiling & MNLI Inference"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        for l in range(NUM_LAYERS):
            for sub_key in ["gate", "up", "down"]:
                t = current_acts[l][sub_key]
                if t is not None:
                    pooled = t.squeeze(0).mean(dim=0).numpy()
                    layer_trajectories[l][sub_key].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

acts_all = {
    l: {
        "gate": np.stack(layer_trajectories[l]["gate"]),
        "up":   np.stack(layer_trajectories[l]["up"]),
        "down": np.stack(layer_trajectories[l]["down"]),
    }
    for l in range(NUM_LAYERS)
}

baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Captured tri-hook activation matrices across all {NUM_LAYERS} layers (78 profiles).")

Simultaneous Tri-Hook Profiling & MNLI Inference: 100%|██████████| 150/150 [00:13<00:00, 10.73it/s]



Uncompressed Baseline Accuracy: 50.67%
Captured tri-hook activation matrices across all 26 layers (78 profiles).
time: 14.17s
cummulative_time: 25.15s


## Step 2: Adaptive Rank Selection via Mode Unfoldings & Adam GD

We define:
1. `compute_adaptive_ranks(T, tau)`: Computes SVD of mode unfoldings and selects bespoke $(R_1, R_2, R_3)$ reaching cumulative energy cutoff $\tau$.
2. `optimize_tucker_gd(T, ranks)`: Refines bespoke core and factors with PyTorch Adam (35 iterations, $lr=10^{-3}$) on CPU.

In [5]:
# =====================================================================
# STEP 5: Define Adaptive Rank Selection & Tucker Adam GD
# =====================================================================
def compute_adaptive_ranks(T, tau, min_ranks=[2, 20, 50]):
    """
    Computes bespoke Tucker ranks (R1, R2, R3) for tensor T by thresholding
    the cumulative singular value energy of each mode unfolding matrix at tau.
    """
    s1 = torch.linalg.svdvals(tl.unfold(T, 0))
    s2 = torch.linalg.svdvals(tl.unfold(T, 1))
    s3 = torch.linalg.svdvals(tl.unfold(T, 2))

    e1 = torch.cumsum(s1**2, dim=0) / torch.sum(s1**2)
    e2 = torch.cumsum(s2**2, dim=0) / torch.sum(s2**2)
    e3 = torch.cumsum(s3**2, dim=0) / torch.sum(s3**2)

    r1 = max(min_ranks[0], int((e1 >= tau).nonzero()[0].item()) + 1)
    r2 = max(min_ranks[1], int((e2 >= tau).nonzero()[0].item()) + 1)
    r3 = max(min_ranks[2], int((e3 >= tau).nonzero()[0].item()) + 1)

    # Bound by tensor dimensions
    r1 = min(r1, T.shape[0])
    r2 = min(r2, T.shape[1])
    r3 = min(r3, T.shape[2])

    return [r1, r2, r3]

def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device="cpu"):
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

print("Adaptive rank discovery and Adam GD optimizer defined.")

Adaptive rank discovery and Adam GD optimizer defined.
time: 0.00s
cummulative_time: 25.16s


## Step 3: Independent Sublayer DBSCAN Pre-Clustering

Every layer $l \in [0 \dots 25]$ independently clusters `gate_proj`, `up_proj`, and `down_proj` into top 6 uniform slices ($2,400$ coords) and quarantines superweights in FP32.

In [6]:
# =====================================================================
# STEP 6: Independent Pre-Clustering for All 78 Submodules
# =====================================================================
CHUNK_SIZE = 400
NUM_CHUNKS = 6

def cluster_submodule(acts_matrix, weight_tensor, is_col=False):
    v = np.mean(acts_matrix, axis=0)
    std_v = np.std(v)
    eps = max(0.04, float(std_v * 0.18))

    db = DBSCAN(eps=eps, min_samples=30, metric="euclidean")
    labels = db.fit_predict(v.reshape(-1, 1))

    max_mags = np.max(np.abs(acts_matrix), axis=0)
    variances = np.var(acts_matrix, axis=0)
    super_mask = (labels == -1) | (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    chunk_list = []
    for lab in unique_labels:
        c_idx = np.where((labels == lab) & (~super_mask))[0]
        if len(c_idx) == 0:
            continue
        sorted_idx = c_idx[np.argsort(v[c_idx])]
        num_full = len(sorted_idx) // CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_idx[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(chunk_list) >= NUM_CHUNKS:
                break
        if len(chunk_list) >= NUM_CHUNKS:
            break

    if len(chunk_list) < NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [i for i in range(len(v)) if i not in assigned and not super_mask[i]]
        needed = NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= CHUNK_SIZE:
                chunk_list.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]

    active_coords = np.concatenate(chunk_list)

    if is_col:
        T = torch.stack([weight_tensor[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        T = torch.stack([weight_tensor[c, :].float().cpu() for c in chunk_list], dim=0)

    return {
        "tensor": T,
        "chunk_list": chunk_list,
        "active_coords": active_coords,
        "super_indices": super_indices,
        "is_col": is_col,
    }

print("Running independent DBSCAN clustering across all 26 layers...")
layer_submodule_data = {}

for l in range(NUM_LAYERS):
    layer_submodule_data[l] = {
        "gate_proj": cluster_submodule(acts_all[l]["gate"], W_orig_all[l]["gate_proj"], is_col=False),
        "up_proj":   cluster_submodule(acts_all[l]["up"],   W_orig_all[l]["up_proj"],   is_col=False),
        "down_proj": cluster_submodule(acts_all[l]["down"], W_orig_all[l]["down_proj"], is_col=True),
    }

print(f"Pre-clustering complete for all {NUM_LAYERS} layers (78 independent clusterings).")

Running independent DBSCAN clustering across all 26 layers...
Pre-clustering complete for all 26 layers (78 independent clusterings).
time: 14.80s
cummulative_time: 39.96s


## Step 4: Multi-Tier Adaptive Energy Compression & Downstream Evaluation

We evaluate 3 adaptive energy cutoffs:
1. **Adaptive Sweet Spot ($\tau = 0.65$)**: Targets ~76–78% reconstruction error across every individual tensor.
2. **Adaptive High-Fidelity ($\tau = 0.70$)**: Targets ~71–73% reconstruction error.
3. **Adaptive Aggressive ($\tau = 0.60$)**: Targets ~80–82% reconstruction error.

In [7]:
# =====================================================================
# STEP 7: Execute Adaptive Compression & Downstream Evaluation
# =====================================================================
adaptive_tiers = [
    {"name": "Adaptive Sweet Spot (tau=0.65)", "tau": 0.65},
    {"name": "Adaptive High-Fidelity (tau=0.70)", "tau": 0.70},
    {"name": "Adaptive Aggressive (tau=0.60)", "tau": 0.60},
]

tier_benchmarks = []

for tier in adaptive_tiers:
    tier_name = tier["name"]
    tau = tier["tau"]

    print(f"\n{'='*100}")
    print(f"Running Adaptive Compression for: {tier_name} (tau={tau})")
    print(f"{'='*100}")

    tier_gate_errs, tier_up_errs, tier_down_errs = [], [], []
    r2_vals, r3_vals = [], []
    total_params_saved = 0

    # 1. Adaptively factorize and inject across all 26 layers
    for l in range(NUM_LAYERS):
        lmod = model.model.layers[l].mlp
        sdata_layer = layer_submodule_data[l]

        for sub_name in ["gate_proj", "up_proj", "down_proj"]:
            sdata = sdata_layer[sub_name]
            T = sdata["tensor"]

            # Compute bespoke ranks dynamically based on this specific tensor's energy decay
            bespoke_ranks = compute_adaptive_ranks(T, tau=tau)
            r2_vals.append(bespoke_ranks[1])
            r3_vals.append(bespoke_ranks[2])

            cg, fg, T_recon, err = optimize_tucker_gd(
                T, ranks=bespoke_ranks, num_steps=35, lr=1e-3, device="cpu"
            )

            if sub_name == "gate_proj": tier_gate_errs.append(err)
            elif sub_name == "up_proj":  tier_up_errs.append(err)
            elif sub_name == "down_proj": tier_down_errs.append(err)

            orig_p = T.numel()
            comp_p = cg.numel() + sum(f.numel() for f in fg)
            total_params_saved += (orig_p - comp_p)

            # Live injection
            mod_ref = getattr(lmod, sub_name)
            orig_w = W_orig_all[l][sub_name]
            mod_ref.weight.data = orig_w.clone().to(model.device)

            if sdata["is_col"]:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[:, c] = T_recon[k].T.to(device=model.device, dtype=mod_ref.weight.dtype)
                mod_ref.weight.data[:, sdata["super_indices"]] = orig_w[:, sdata["super_indices"]].to(model.device)
            else:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[c, :] = T_recon[k].to(device=model.device, dtype=mod_ref.weight.dtype)
                mod_ref.weight.data[sdata["super_indices"], :] = orig_w[sdata["super_indices"], :].to(model.device)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_gate_err = np.mean(tier_gate_errs) * 100
    mean_up_err   = np.mean(tier_up_errs) * 100
    mean_down_err = np.mean(tier_down_errs) * 100
    r2_range = f"[{min(r2_vals)}-{max(r2_vals)}]"
    r3_range = f"[{min(r3_vals)}-{max(r3_vals)}]"

    print(f"Layer Factorization Complete.")
    print(f"  Bespoke Ranks: Mode-2 Range={r2_range}, Mode-3 Range={r3_range}")
    print(f"  Mean Recon Errors: gate={mean_gate_err:.1f}%, up={mean_up_err:.1f}%, down={mean_down_err:.1f}%")
    print(f"  Total Parameters Eliminated: {total_params_saved:,}")

    # 2. Evaluate on GLUE MNLI
    preds, gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            preds.append(pred_label)
            gts.append(sample["label"])

    acc = accuracy_score(gts, preds)
    delta = acc - baseline_accuracy

    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Total Params Cut:    {total_params_saved:,}")

    tier_benchmarks.append({
        "Variant": tier_name,
        "Tau": tau,
        "R2_Range": r2_range,
        "R3_Range": r3_range,
        "Mean_Gate_Err": round(mean_gate_err, 2),
        "Mean_Up_Err": round(mean_up_err, 2),
        "Mean_Down_Err": round(mean_down_err, 2),
        "Params_Eliminated": total_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore pristine model weights across all 26 layers
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        getattr(model.model.layers[l].mlp, sub_name).weight.data = W_orig_all[l][sub_name].clone().to(model.device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRestored all 26 layers to pristine weights.")


Running Adaptive Compression for: Adaptive Sweet Spot (tau=0.65) (tau=0.65)
Layer Factorization Complete.
  Bespoke Ranks: Mode-2 Range=[195-217], Mode-3 Range=[309-400]
  Mean Recon Errors: gate=77.4%, up=78.8%, down=78.1%
  Total Parameters Eliminated: 152,395,068


Evaluating Adaptive Sweet Spot (tau=0.65): 100%|██████████| 150/150 [00:04<00:00, 31.66it/s]



Result for Adaptive Sweet Spot (tau=0.65):
  Downstream Accuracy: 42.00% (Δ vs Baseline: -8.67%)
  Total Params Cut:    152,395,068

Running Adaptive Compression for: Adaptive High-Fidelity (tau=0.70) (tau=0.7)
Layer Factorization Complete.
  Bespoke Ranks: Mode-2 Range=[218-239], Mode-3 Range=[357-452]
  Mean Recon Errors: gate=72.8%, up=71.5%, down=72.1%
  Total Parameters Eliminated: 135,443,950


Evaluating Adaptive High-Fidelity (tau=0.70): 100%|██████████| 150/150 [00:04<00:00, 33.63it/s]



Result for Adaptive High-Fidelity (tau=0.70):
  Downstream Accuracy: 40.00% (Δ vs Baseline: -10.67%)
  Total Params Cut:    135,443,950

Running Adaptive Compression for: Adaptive Aggressive (tau=0.60) (tau=0.6)
Layer Factorization Complete.
  Bespoke Ranks: Mode-2 Range=[174-197], Mode-3 Range=[266-354]
  Mean Recon Errors: gate=79.8%, up=81.1%, down=80.5%
  Total Parameters Eliminated: 162,173,932


Evaluating Adaptive Aggressive (tau=0.60): 100%|██████████| 150/150 [00:04<00:00, 32.83it/s]



Result for Adaptive Aggressive (tau=0.60):
  Downstream Accuracy: 42.00% (Δ vs Baseline: -8.67%)
  Total Params Cut:    162,173,932

Restored all 26 layers to pristine weights.
time: 581.64s
cummulative_time: 621.61s


## Step 5: Comparative Synthesis & Artifact Export

We tabulate the multi-tier performance comparisons against baseline and export metrics to `artifacts/13_all_layers_adaptive_results.json`.

In [8]:
# =====================================================================
# STEP 8: Comparative Synthesis & Artifact Export
# =====================================================================
total_mlp_params_orig = NUM_LAYERS * 3 * (D_IN * D_OUT)  # 621,084,672

print("=" * 120)
print(f"{'Variant':<42} | {'Ranks Range (M2, M3)':<22} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<11} | {'Accuracy':<9} | {'Delta':<8}")
print("=" * 120)
print(f"{'Baseline (Uncompressed)':<42} | {'Full [400, 1152]':<22} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<11} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for r in tier_benchmarks:
    rank_str = f"M2:{r['R2_Range']} M3:{r['R3_Range']}"
    print(f"{r['Variant']:<42} | {rank_str:<22} | {r['Mean_Gate_Err']:>6.2f}% | {r['Mean_Up_Err']:>5.2f}% | {r['Mean_Down_Err']:>6.2f}% | {r['Params_Eliminated']:>10,} | {r['Accuracy']:>7.2f}% | {r['Delta']:>+6.2f}%")

print("=" * 120)

os.makedirs("artifacts", exist_ok=True)
output_payload = {
    "experiment": "13_all_layers_adaptive_tucker",
    "target_model": model_id,
    "num_layers": NUM_LAYERS,
    "num_projections": NUM_LAYERS * 3,
    "total_mlp_params_orig": total_mlp_params_orig,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "tiers": tier_benchmarks,
    "timings": NOTEBOOK_TIMINGS,
}

with open("artifacts/13_all_layers_adaptive_results.json", "w") as f:
    json.dump(output_payload, f, indent=2)

print(f"Saved benchmark results to artifacts/13_all_layers_adaptive_results.json")

Variant                                    | Ranks Range (M2, M3)   | Gate Err  | Up Err   | Down Err  | Params Cut  | Accuracy  | Delta   
Baseline (Uncompressed)                    | Full [400, 1152]       | 0.00%     | 0.00%    | 0.00%     | 0           |   50.67% | +0.00%  
Adaptive Sweet Spot (tau=0.65)             | M2:[195-217] M3:[309-400] |  77.37% | 78.77% |  78.14% | 152,395,068 |   42.00% |  -8.67%
Adaptive High-Fidelity (tau=0.70)          | M2:[218-239] M3:[357-452] |  72.81% | 71.52% |  72.05% | 135,443,950 |   40.00% | -10.67%
Adaptive Aggressive (tau=0.60)             | M2:[174-197] M3:[266-354] |  79.79% | 81.07% |  80.54% | 162,173,932 |   42.00% |  -8.67%
Saved benchmark results to artifacts/13_all_layers_adaptive_results.json
time: 0.00s
cummulative_time: 621.62s
